# 推理忠实性人工标注 Notebook

> 论文 §5.4 数据来源。验证 `cot_full` 推理链中引用的 "证据" 是否真实存在于对话文本，回答 *"模型是真推理还是事后合理化"*。

## 工作流（三阶段）

1. **生成模板**：从 `results/preds_cot_full_taxi_full.json` 抽 50 条 `jga=True` 的样本，导出 `results/faithfulness_template.csv`（含 `reasoning_faithful` / `note` 空列）
2. **人工标注**（在 Excel / VS Code / 任何支持 CSV 的工具中）：
   - `reasoning_faithful = TRUE` 当且仅当推理链中每次 `[turn N]` 引用对应的 turn **真实包含**所引用内容
   - 存在任何虚构引用 → `FALSE`
   - 把已标注文件另存为 `results/faithfulness_annotation.csv`（保留原列 + 填好的两列）
3. **统计分析**：本 notebook 最后一个 cell 加载 `faithfulness_annotation.csv`，输出四象限频率 + 可视化

## 标注规则速查

| 情形 | reasoning_faithful |
|---|---|
| 推理引用了 `[turn 3]` 说"user mentioned X"，turn 3 user 确实说了 X | TRUE |
| 推理只引用了概念而无 turn 索引但叙述与对话吻合 | TRUE |
| 推理引用 `[turn 5]` 但 turn 5 并未提到所述内容 | FALSE |
| 推理凭空捏造了对话不存在的事实 | FALSE |
| 推理给出了空白 `<reasoning>` 块或仅复述 schema | FALSE（无法判定为忠实）|

## 阶段 1：生成标注模板

In [ ]:
from __future__ import annotations

import json
import random
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from evaluator import eval_single  # type: ignore

RESULTS_DIR = PROJECT_ROOT / 'results'
PRED_PATH = RESULTS_DIR / 'preds_cot_full_taxi_full.json'
TEMPLATE_PATH = RESULTS_DIR / 'faithfulness_template.csv'
ANNOTATED_PATH = RESULTS_DIR / 'faithfulness_annotation.csv'

N_SAMPLES = 50
RANDOM_SEED = 42
TARGET_DOMAIN = 'taxi'

print(f'Pred file       : {PRED_PATH}')
print(f'Template output : {TEMPLATE_PATH}')
print(f'Annotated input : {ANNOTATED_PATH}')
print(f'Exists: pred={PRED_PATH.exists()}  template={TEMPLATE_PATH.exists()}  annotated={ANNOTATED_PATH.exists()}')

In [ ]:
def _format_history(history: list[tuple[str, str]]) -> str:
    """Render dialogue history with turn indices so annotators can verify reasoning citations."""
    lines = []
    turn_idx = 0
    for role, text in history:
        if role == 'user':
            turn_idx += 1
            lines.append(f'[turn {turn_idx}] User: {text}')
        else:
            lines.append(f'           System: {text}')
    return '\n'.join(lines)


def build_template(pred_path: Path, n: int, seed: int, target_domain: str) -> pd.DataFrame:
    """Sample `n` jga=true predictions and write them to a CSV template for annotation."""
    with open(pred_path, encoding='utf-8') as f:
        preds = json.load(f)

    correct = []
    for item in preds:
        res = eval_single(item['gold_belief'], item['pred_belief'], target_domain=target_domain)
        if res['jga']:
            correct.append(item)

    print(f'Total preds      : {len(preds)}')
    print(f'jga=true (taxi)  : {len(correct)}')
    if len(correct) < n:
        raise RuntimeError(f'Need {n} jga=true samples but only found {len(correct)}.')

    rng = random.Random(seed)
    sampled = rng.sample(correct, n)

    rows = []
    for i, item in enumerate(sampled, start=1):
        # history may not be in pred file (only dial_id/turn_id/belief is). We attach if present.
        history_str = _format_history(item.get('history', [])) if item.get('history') else ''
        rows.append({
            'sample_idx': i,
            'dial_id': item['dial_id'],
            'turn_id': item['turn_id'],
            'history': history_str,
            'reasoning': item.get('reasoning', ''),
            'gold_belief': json.dumps(item.get('gold_belief', {}), ensure_ascii=False),
            'pred_belief': json.dumps(item.get('pred_belief', {}), ensure_ascii=False),
            'reasoning_faithful': '',  # to fill: TRUE / FALSE
            'note': '',
        })
    return pd.DataFrame(rows)

In [ ]:
# history 字段 ablation_runner 不会写入预测，需要从 test_taxi.json 关联回填
TEST_TAXI_PATH = PROJECT_ROOT / 'data' / 'processed' / 'test_taxi.json'

def _load_history_index(test_path: Path) -> dict[tuple[str, int], list]:
    with open(test_path, encoding='utf-8') as f:
        data = json.load(f)
    return {(d['dial_id'], d['turn_id']): d['history'] for d in data}


def build_template_with_history(pred_path: Path, n: int, seed: int, target_domain: str) -> pd.DataFrame:
    hist_idx = _load_history_index(TEST_TAXI_PATH)
    with open(pred_path, encoding='utf-8') as f:
        preds = json.load(f)
    correct = []
    for item in preds:
        res = eval_single(item['gold_belief'], item['pred_belief'], target_domain=target_domain)
        if res['jga']:
            correct.append(item)
    print(f'Total preds      : {len(preds)}')
    print(f'jga=true (taxi)  : {len(correct)}')
    if len(correct) < n:
        raise RuntimeError(f'Need {n} jga=true samples but only found {len(correct)}.')
    rng = random.Random(seed)
    sampled = rng.sample(correct, n)
    rows = []
    for i, item in enumerate(sampled, start=1):
        history = hist_idx.get((item['dial_id'], item['turn_id']), [])
        rows.append({
            'sample_idx': i,
            'dial_id': item['dial_id'],
            'turn_id': item['turn_id'],
            'history': _format_history(history),
            'reasoning': item.get('reasoning', ''),
            'gold_belief': json.dumps(item.get('gold_belief', {}), ensure_ascii=False),
            'pred_belief': json.dumps(item.get('pred_belief', {}), ensure_ascii=False),
            'reasoning_faithful': '',
            'note': '',
        })
    return pd.DataFrame(rows)

# 取消下面这行注释以生成模板（需要先跑完 Phase 2B 全量实验）
# df_template = build_template_with_history(PRED_PATH, N_SAMPLES, RANDOM_SEED, TARGET_DOMAIN)
# df_template.to_csv(TEMPLATE_PATH, index=False, encoding='utf-8-sig')
# print(f'\n✅ 模板已写入 {TEMPLATE_PATH}')
# df_template.head(3)

---

## 阶段 2：人工标注（外部完成）

1. 用 Excel 打开 `results/faithfulness_template.csv`（utf-8-sig 兼容 Excel 中文显示）
2. 对每行：
   - 阅读 `history` 列（对话上下文，按 `[turn N]` 标记）
   - 阅读 `reasoning` 列（模型四步推理链）
   - 核验：推理中每次引用 `[turn X]` 或 "the user said Y"，确认对应 turn 真实包含
   - 在 `reasoning_faithful` 列填 `TRUE` 或 `FALSE`
   - 可选在 `note` 列写一句话说明
3. 另存为 `results/faithfulness_annotation.csv`（覆盖或新建均可）

**预估工时**：2–3 小时（50 条 × ~3 分钟/条）

---

## 阶段 3：统计分析（标注完成后运行）

In [ ]:
def load_annotated(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding='utf-8-sig')
    df['reasoning_faithful'] = (
        df['reasoning_faithful'].astype(str).str.upper().str.strip()
        .replace({'TRUE': True, 'FALSE': False, '1': True, '0': False,
                  'T': True, 'F': False})
    )
    return df


def quadrant_stats(df: pd.DataFrame, gold_jga: bool = True) -> pd.DataFrame:
    """All sampled rows are jga=true by construction, so the 2×2 reduces to a 1×2 split.
    For the full 四象限 we'd also sample jga=false predictions — left as future work in §5.4."""
    counts = df['reasoning_faithful'].value_counts(dropna=False).to_dict()
    total = len(df)
    tbl = pd.DataFrame({
        'faithful_correct (理想)':    [counts.get(True, 0)],
        'unfaithful_correct (侥幸)':  [counts.get(False, 0)],
        'unparseable / blank':       [counts.get('', 0) + counts.get('NAN', 0)],
        'total':                      [total],
    })
    pct = (tbl.iloc[0] / total * 100).round(1)
    tbl.loc['percent'] = pct
    return tbl


# 取消注释以运行统计（前提：阶段 2 已完成）
# df_ann = load_annotated(ANNOTATED_PATH)
# print(quadrant_stats(df_ann))
# print('\n样例（虚假推理但正确）:')
# unfaithful = df_ann[df_ann['reasoning_faithful'] == False]
# for _, r in unfaithful.head(3).iterrows():
#     print(f"\n--- {r['dial_id']} turn={r['turn_id']} ---\n{r['note']}")

---

## 论文写作 hook（§5.4 表格生成）

In [ ]:
def write_section_5_4_markdown(df: pd.DataFrame, out_path: Path) -> None:
    total = len(df)
    faithful = (df['reasoning_faithful'] == True).sum()
    unfaithful = (df['reasoning_faithful'] == False).sum()
    blank = total - faithful - unfaithful
    md = [
        '# §5.4 推理忠实性分析\n',
        f'> 数据来源：50 条 `cot_full` 在 taxi 子集上 `jga=True` 的随机样本（seed={RANDOM_SEED}）\n',
        '## 标注分布\n',
        '| 类别 | 数量 | 占比 |',
        '|---|---|---|',
        f'| 忠实推理 → 正确（理想情形） | {faithful} | {faithful/total*100:.1f}% |',
        f'| 虚假推理 → 正确（事后合理化） | {unfaithful} | {unfaithful/total*100:.1f}% |',
        f'| 未标注 / 空白 | {blank} | {blank/total*100:.1f}% |',
        f'| **合计** | **{total}** | 100.0% |',
    ]
    out_path.write_text('\n'.join(md) + '\n', encoding='utf-8')
    print(f'§5.4 表格已写入 {out_path}')

# write_section_5_4_markdown(df_ann, RESULTS_DIR / 'faithfulness_summary.md')